# L'aléa décide de la règle d'agrégation

Ce carnet rejoue la section 5.3. La question est simple : la règle qui
mène du score d'un pixel au score d'un bâtiment est-elle la même pour un séisme
et pour un cyclone ?

Deux tables, une par événement :

- `analyse_complete_seuils.csv` — le balayage complet seuil × règle
  d'agrégation sur le séisme ;
- `jamaique_agregations_detail.csv` — l'AUC par règle d'agrégation sur le
  cyclone.

**Une réserve à lire avant les chiffres.** Les deux tables ne portent pas le
même indicateur : la première donne un kappa maximal sur un balayage de seuils,
la seconde une AUC. On ne les compare donc pas entre elles ; on regarde, dans
chacune, l'écart entre règles d'agrégation.

In [ ]:
import os
import pandas as pd

# les carnets se lancent depuis notebooks/ ; les tables sont dans data/results
RESULTATS = os.path.join("..", "data", "results")
TABLES = os.path.join("..", "tables")

def lire(dossier, nom, **kw):
    """Lit une table du depot. Le separateur et la virgule decimale varient
    d'une campagne a l'autre : on laisse pandas les deviner sauf indication."""
    kw.setdefault("sep", None)
    kw.setdefault("engine", "python")
    return pd.read_csv(os.path.join(dossier, nom), **kw)

def presque(a, b, tol=0.0006):
    """Egalite au millieme pres, tolerance par defaut d'un demi-millieme."""
    assert abs(a - b) < tol, "%.4f attendu, %.4f obtenu" % (b, a)
    return True


Sur le séisme, la règle d'agrégation change nettement le résultat.

In [ ]:
s = lire(RESULTATS, "analyse_complete_seuils.csv")
seisme = (s[s.scope == "GLOBAL"].groupby("agg")["kappa"].max()
          .sort_values(ascending=False))
print("Seisme — meilleur kappa par regle d'agregation :")
print(seisme.round(4).to_string())

Sur le cyclone, les règles se tiennent dans un mouchoir de poche.

In [ ]:
j = lire(RESULTATS, "jamaique_agregations_detail.csv")
cyclone = j.groupby("agregation")["auc"].mean().sort_values(ascending=False)
print("Cyclone — AUC moyenne par regle d'agregation :")
print(cyclone.round(4).to_string())
print("\netendue entre la meilleure et la moins bonne regle : %.4f"
      % (cyclone.max() - cyclone.min()))

La conclusion, vérifiée : ce qui départage les règles sur un séisme ne les départage pas sur un cyclone.

In [ ]:
# le constat de l'article : l'ecart entre regles est d'un ordre de grandeur
# different d'un alea a l'autre.
etendue_seisme = seisme.max() - seisme.min()
etendue_cyclone = cyclone.max() - cyclone.min()
print("etendue seisme  (kappa) : %.4f" % etendue_seisme)
print("etendue cyclone (AUC)   : %.4f" % etendue_cyclone)
assert etendue_cyclone < 0.01, "le cyclone devrait etre insensible a la regle"
print("\nVerifie : sur le cyclone, la regle d'agregation ne departage rien.")